# 22 — Entity Resolution, Duplicate Keys, & Merge Conflict Handling
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive guide to entity reconciliation, duplicate key resolution, suffix collision management, and multi-system data integration in Pandas.*

---

## 📌 Executive Summary & Interview Expectations
In distributed data systems (e.g. merging CRM and billing systems), entity identifiers often overlap, conflict, or exist across multiple partitions with contradictory attributes.
This lab tests critical data integration skills:
1. **Vertical Concatenation with Identity Overlaps**: What happens when ID ranges overlap between batches (`subject_id` 4 and 5 in both datasets).
2. **Column Collision & Suffix Management**: Explicitly naming conflicting columns via `suffixes=('_crm', '_billing')` instead of accepting opaque `_x`/`_y` defaults.
3. **1-to-Many Merge Cardinality**: Observing how duplicate keys on one side replicate rows during joins with dimension tables.
4. **Entity Conflict Resolution**: Detecting conflicting attribute values across systems and applying reconciliation business logic.

## 1. Environment Setup & Data Construction

In [1]:
import numpy as np
import pandas as pd

raw_data_1 = {
    'subject_id': ['1', '2', '3', '4', '5'],
    'first_name': ['Alex', 'Amy', 'Allen', 'Alice', 'Ayoung'], 
    'last_name': ['Anderson', 'Ackerman', 'Ali', 'Aoni', 'Atiches']
}

raw_data_2 = {
    'subject_id': ['4', '5', '6', '7', '8'],
    'first_name': ['Billy', 'Brian', 'Bran', 'Bryce', 'Betty'], 
    'last_name': ['Bonder', 'Black', 'Balwner', 'Brice', 'Btisan']
}

raw_data_3 = {
    'subject_id': ['1', '2', '3', '4', '5', '7', '8', '9', '10', '11'],
    'test_id': [51, 15, 15, 61, 16, 14, 15, 1, 61, 16]
}

data1 = pd.DataFrame(raw_data_1)
data2 = pd.DataFrame(raw_data_2)
data3 = pd.DataFrame(raw_data_3)

print("Batch 1 (Subjects 1-5):")
display(data1)
print("Batch 2 (Subjects 4-8):")
display(data2)
print("Batch 3 (Test Scores):")
display(data3.head(4))

Batch 1 (Subjects 1-5):


,subject_id,first_name,last_name
0,1,Alex,Anderson
1,2,Amy,Ackerman
2,3,Allen,Ali
3,4,Alice,Aoni
4,5,Ayoung,Atiches


Batch 2 (Subjects 4-8):


,subject_id,first_name,last_name
0,4,Billy,Bonder
1,5,Brian,Black
2,6,Bran,Balwner
3,7,Bryce,Brice
4,8,Betty,Btisan


Batch 3 (Test Scores):


,subject_id,test_id
0,1,51
1,2,15
2,3,15
3,4,61


## 2. Row Stacking: Row Index Resetting vs Provenance Tracking

### ⚠️ Top Interview Question: What happens to `subject_id` 4 and 5?
- Subjects 4 and 5 appear in **both** `data1` and `data2`.
- When concatenating along rows (`axis=0`), `pd.concat` does NOT deduplicate rows automatically.
- Always use `ignore_index=True` so that row index numbers remain continuous and unique.

In [2]:
# Stacking rows with clean range index
all_data = pd.concat([data1, data2], ignore_index=True)
print(f"Total Rows in all_data: {len(all_data)} (Duplicate subject_ids present: {all_data['subject_id'].duplicated().any()})")
display(all_data)

Total Rows in all_data: 10 (Duplicate subject_ids present: True)


,subject_id,first_name,last_name
0,1,Alex,Anderson
1,2,Amy,Ackerman
2,3,Allen,Ali
3,4,Alice,Aoni
4,5,Ayoung,Atiches
5,4,Billy,Bonder
6,5,Brian,Black
7,6,Bran,Balwner
8,7,Bryce,Brice
9,8,Betty,Btisan


## 3. Column Stacking: Positional Alignment Hazards

### 🚨 Production Gotcha: `axis=1` Concatenation Without Alignment
- `pd.concat([data1, data2], axis=1)` stacks tables horizontally by **row index**, NOT by `subject_id`!
- At row 0, `data1` has subject 1, while `data2` has subject 4!
- Stacking side-by-side without aligning keys produces completely corrupted records!

In [3]:
all_data_col = pd.concat([data1, data2], axis=1)
print("Side-by-side concat (Notice index 0 has subject 1 aligned with subject 4!):")
display(all_data_col)

Side-by-side concat (Notice index 0 has subject 1 aligned with subject 4!):


,subject_id,first_name,last_name,subject_id,first_name,last_name
0,1,Alex,Anderson,4,Billy,Bonder
1,2,Amy,Ackerman,5,Brian,Black
2,3,Allen,Ali,6,Bran,Balwner
3,4,Alice,Aoni,7,Bryce,Brice
4,5,Ayoung,Atiches,8,Betty,Btisan


## 4. Merging with Dimension Table & Cardinality Expansion

When merging `all_data` with `data3`:
- Since `all_data` contains duplicate `subject_id` ('4' and '5'), each duplicate row matches `data3['subject_id']`.
- This expands the row count for those keys!

In [4]:
# Merging all_data with data3 on subject_id
merged_tests = pd.merge(all_data, data3, on="subject_id")
print("Merged Tests Sample (Notice duplicate entries for subjects 4 and 5):")
display(merged_tests[merged_tests["subject_id"].isin(["4", "5"])])

Merged Tests Sample (Notice duplicate entries for subjects 4 and 5):


,subject_id,first_name,last_name,test_id
3,4,Alice,Aoni,61
4,5,Ayoung,Atiches,16
5,4,Billy,Bonder,61
6,5,Brian,Black,16


## 5. Handling Column Collisions: Explicit Suffixes

### 💡 Production Standard: Never Accept Default `_x` and `_y`
Default suffixes (`first_name_x`, `first_name_y`) obscure data lineage. Always declare descriptive suffixes:
```python
suffixes=('_system_a', '_system_b')
```

In [5]:
# 1. Inner Join: Overlapping subjects between data1 and data2
inner_conflict = pd.merge(
    data1,
    data2,
    on="subject_id",
    how="inner",
    suffixes=("_batch1", "_batch2")
)
print("Inner Join with Explicit Suffixes:")
display(inner_conflict)

# 2. Full Outer Join: Complete universe of records across both batches
outer_all = pd.merge(
    data1,
    data2,
    on="subject_id",
    how="outer",
    suffixes=("_batch1", "_batch2")
)
print("\nOuter Join showing all subjects:")
display(outer_all)

Inner Join with Explicit Suffixes:


,subject_id,first_name_batch1,last_name_batch1,first_name_batch2,last_name_batch2
0,4,Alice,Aoni,Billy,Bonder
1,5,Ayoung,Atiches,Brian,Black



Outer Join showing all subjects:


,subject_id,first_name_batch1,last_name_batch1,first_name_batch2,last_name_batch2
0,1,Alex,Anderson,NaN,NaN
1,2,Amy,Ackerman,NaN,NaN
2,3,Allen,Ali,NaN,NaN
3,4,Alice,Aoni,Billy,Bonder
4,5,Ayoung,Atiches,Brian,Black
5,6,NaN,NaN,Bran,Balwner
6,7,NaN,NaN,Bryce,Brice
7,8,NaN,NaN,Betty,Btisan


---
## 🎯 6. Technical Interview Corner: Tricky Questions & Drills

### Q1: What happens if `subject_id` in `data1` is `string` and in `data3` is `int`?
**Answer**:
- In modern Pandas, merging across mismatched types (e.g. `str` and `int64`) raises a **`ValueError: You are trying to merge on object and int64 columns`**.
- In legacy Pandas, it would silently return an empty DataFrame (0 rows), leading to silent pipeline failures!
- **Rule**: Always standardize key dtypes before joining: `df['id'] = df['id'].astype(str)`.

---

### Q2: Advanced Interview Coding Challenge: Reconciliation & Conflict Resolution
**Challenge**:
In `outer_all`, some subjects exist only in Batch 1, some only in Batch 2, and subjects 4 and 5 have **conflicting names** in both batches!
Write a production reconciliation function that:
1. Prefers `Batch 2` values when a conflict occurs.
2. Falls back to `Batch 1` when `Batch 2` is missing.
3. Produces a clean, single `first_name` and `last_name` without `_batch1`/`_batch2` suffixes!

In [6]:
# Interview Solution: Robust Coalesce / Conflict Arbitration
reconciled = outer_all.assign(
    first_name=lambda df: df["first_name_batch2"].combine_first(df["first_name_batch1"]),
    last_name=lambda df: df["last_name_batch2"].combine_first(df["last_name_batch1"])
)[["subject_id", "first_name", "last_name"]].sort_values("subject_id", key=lambda s: s.astype(int)).reset_index(drop=True)

print("Reconciled Single-Source-of-Truth Entity Table:")
display(reconciled)

Reconciled Single-Source-of-Truth Entity Table:


,subject_id,first_name,last_name
0,1,Alex,Anderson
1,2,Amy,Ackerman
2,3,Allen,Ali
3,4,Billy,Bonder
4,5,Brian,Black
5,6,Bran,Balwner
6,7,Bryce,Brice
7,8,Betty,Btisan
